In [1]:
# # скачиваем уже обработанную версию ямбды 500m, в которой только лайки

!uv pip install -q gdown
!gdown --id 1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS -O dataset.zip
!unzip -q dataset.zip

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS
From (redirected): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS&confirm=t&uuid=b66f25dc-66d9-43c2-b117-5b8396bfe499
To: /content/dataset.zip
100% 356M/356M [00:06<00:00, 57.1MB/s] 


### 1. Подготовка данных

**Задача:**
1) Считать данные (взаимодействия, эмбеддинги, метаданные).  
2) Оставить только взаимодействия, для которых есть эмбеддинги.  
3) Сделать core фильтрацию: оставить только айтемы с ≥5 взаимодействиями (в ДЗ мы делаем это исключительно для удобства и скорости, в реальной работе так делать не стоит)
4) Поджоинить метаданные (артисты треков) ко всем взаимодействиям. Ремарка: здесь у нас если у одного трека несколько артистов произойдет дублирование прослушивание - в рамках ДЗ мы ничего с этим не делаем, но в реальной жизни такого допускать нельзя. 
5) Сделать train-test split: последнюю неделю положить в тест.  
6) Ограничить тест юзерами, у которых есть взаимодействия в трейне.  
7) Подготовить для оценки качества `test_targets: Dict[uid, List[item_id]]`.
8) Оставить только эмбеддинги для core айтемов (остальные пофильтровать).

После этого блока должны существовать `train`, `test`, `embeddings`, `artists`, `test_targets`.

Для этого блока полезны как минимум следующие методы:
* `pl.read_parquet` - для чтения данных
* `.filter, .value_counts` помогут сделать core-фильтрацию
* `df.join(...)` - при джойне метаданных надо использовать `how='left'`, а не `how='inner'`
* `df.join(other, on=some_key, how='semi')` - режим `semi` используется для фильтраций (оставить только те строки из исходного датафрейма, ключ которых присутствует во второй таблице)

In [2]:
from typing import Dict, List
import os

import numpy as np
import polars as pl

# Пути к данным (ожидается, что они лежат рядом с ноутбуком)
DATA_DIR = "."
PATH_INTERACTIONS = os.path.join(DATA_DIR, "interactions.parquet")
PATH_EMBEDDINGS = os.path.join(DATA_DIR, "embeddings.parquet")
PATH_ARTISTS = os.path.join(DATA_DIR, "artists.parquet")

# Глобальные параметры
TOPK = 100
CORE_MIN_INTERACTIONS_PER_ITEM = 5
TEST_INTERVAL_SECONDS = 7 * 24 * 60 * 60

# Для воспроизводимости
np.random.seed(42)

interactions = pl.read_parquet(PATH_INTERACTIONS)
embeddings = pl.read_parquet(PATH_EMBEDDINGS)
artists = pl.read_parquet(PATH_ARTISTS)

interactions = interactions.join(embeddings, on="item_id", how="semi")

count = interactions.group_by('item_id').len()

interactions = (interactions.join(count, on='item_id', how='left')
                .filter(pl.col('len') >= CORE_MIN_INTERACTIONS_PER_ITEM))

interactions = interactions.join(artists, on='item_id', how='semi')

end = interactions['timestamp'].max()

train = interactions.filter(pl.col('timestamp') < end - TEST_INTERVAL_SECONDS)
test = interactions.filter(pl.col('timestamp') >= end - TEST_INTERVAL_SECONDS)
test = test.filter(pl.col('uid').is_in(train['uid'].implode()))

test_targets = {
    i[0]: i[1]
    for i in test.group_by('uid').agg('item_id').rows()
}

embeddings = embeddings.filter(pl.col('item_id').is_in(interactions['item_id'].implode()))


In [3]:
# Запуск автопроверок (для них необходим файл tests.py)
import tests

tests.check_data_split(train=train, test=test, test_targets=test_targets)

All good! :)


### 2. Оценка качества

#### 2.1 Определения метрик

2.1.1 Пусть для пользователя $u$:

* $G_u \subset \mathcal{I}$ — множество релевантных айтемов (ground truth)
* $R_u = (r_{u,1}, \dots, r_{u,K})$ — упорядоченный список рекомендаций длины $K$

Обозначим индикатор релевантности $I_{u,k} = [ r_{u, k} \in G_u]$. В простонародье его еще часто называют `hits`.

2.1.2 **Hitrate@K** равен единичке, если мы угадали в topK хотя бы один релевантный айтем:
* $
\text{Hitrate@K} = \frac{1}{|U|}
\sum_{u \in U}
\left[ \sum_{k=1}^{K} I_{u,k} > 0 \right]
$

2.1.3 **Recall@K** оценивает долю угаданных релевантных айтемов (от всех релевантных айтемов):
* $
\text{Recall@K} = \frac{1}{|U|}
\sum_{u \in U}
\frac{
\sum_{k=1}^{K} I_{u,k}
}{
\min(|G_u|, K)
}
$

2.1.4 Для подсчета **nDCG@K** нужно сначала посчитать **DCG@K**, затем посчитать **iDCG@K** (DCG в случае идеального ранжирования), затем одно поделить на другое:
* $
\text{DCG@K}(u) = \sum_{k=1}^{K}
\frac{I_{u,k}}{\log_2(k+1)}
$
* $
\text{iDCG@K}(u) = \sum_{k=1}^{\min(|G_u|,K)}
\frac{1}{\log_2(k+1)}
$
* $
\text{nDCG@K} = \frac{1}{|U|}
\sum_{u \in U}
\frac{\text{DCG@K}(u)}{\text{iDCG@K}(u)}
$

2.1.5 **Coverage@K** - это число уникальных айтемов во всех рекомендациях, деленное на размер каталога:

* $
\text{Coverage@K} = \frac{|\bigcup_{u \in U} R_u|}{|\mathcal{I}_{train}|},
$ где $\mathcal{I}_{train}$ — каталог айтемов в train.
* в качестве размера каталога используем количество айтемов, которые нам доступны для рекомендации на момент рекомендации (то есть количество уникальных айтемов в `train`)

#### 2.2 Что нужно сделать
Реализуйте функции:
- `get_metrics(targets, candidates, topk) -> dict(hitrate, recall, ndcg)`
- `evaluate(targets_by_user, candidates_by_user, catalog_size, topk) -> dict(hitrate, recall, ndcg, coverage)`

**Важно:** 
* `candidates[uid]` должен иметь длину ровно `topk`
* при подсчете метрики нужно дедуплицировать позитивы; это влияет на значение, на которое мы делим


In [4]:
import math


def get_metrics(targets: List[int], candidates: List[int], topk: int) -> Dict[str, float]:
    return {
        'hitrate': min(1, sum(1 if i in targets else 0 for i in candidates)),
        'recall': sum(1 if i in targets else 0 for i in candidates)
                  / min(topk, len(targets)),
        'ndcg': sum(1.0 / math.log2(i + 2) if j in targets else 0
                    for i, j in enumerate(candidates)) /
                sum(1 / math.log2(j + 2) for j in range(min(topk, len(targets))))
    }


def evaluate(
        targets: Dict[int, List[int]],
        candidates: Dict[int, List[int]],
        catalog_size: int,
        topk: int = 100,
) -> Dict[str, float]:
    hitrates = []
    recalls = []
    ndcgs = []

    s = set()

    for i in candidates:
        s = s.union(set(candidates[i]))
        metrics = get_metrics(targets[i], candidates[i], topk)
        hitrates.append(metrics['hitrate'])
        recalls.append(metrics['recall'])
        ndcgs.append(metrics['ndcg'])

    return {
        'hitrate': sum(hitrates) / len(hitrates),
        'recall': sum(recalls) / len(recalls),
        'ndcg': sum(ndcgs) / len(ndcgs),
        'coverage': len(s) / catalog_size
    }

In [5]:
tests.check_metrics(get_metrics=get_metrics, evaluate=evaluate)

All good! :)


### 3. Токенизация

Нужно построить таблицу item_to_token_id.

#### Что делаем

1. Берём `train['item_id']`
2. Считаем популярности `(value_counts)`
3. Сортируем:
* `count` ↓
* `item_id` ↑ (для детерминизма)
4. Берём `head(VOCAB_ITEMS)`
5. Назначаем `token_id = row_index + 1`
(0 зарезервирован под padding)
6. Оставляем только `['item_id', 'token_id']`

In [6]:
VOCAB_ITEMS = 30_000

item_to_token_id = (
    train
    .select("item_id")  # берём только item_id (проще и быстрее)
    .group_by('item_id').len().rename({'len': 'count'})
    .sort('item_id')
    .reverse()
    .sort('count')
    .reverse()
    .head(VOCAB_ITEMS)
    .with_row_index(name='token_id')
    .with_columns(pl.col('token_id') + 1)
    .select(('item_id', 'token_id'))
)

In [7]:
tests.test_tokenizer(item_to_token_id)

### 4. Обработка историй пользователя

Готовим последовательности для обучения.

#### Что нужно сделать

1. Оставить только те события, где `item_id` есть в `item_to_token_id`
   → `inner join`

2. Отсортировать по `timestamp`

3. Для каждого `uid` оставить последние `MAX_EVENTS_PER_USER` событий
   → `.group_by(...).tail(...)`

4. Построить `train_histories`:
   для каждого пользователя список `token_id`,
   где в начале добавлен `BOS = 0`


In [8]:
MAX_EVENTS_PER_USER = 100  # включая BOS

# 1. Фильтрация + сортировка
train_events = (
    train
    .join(item_to_token_id, on='item_id', how='inner')
    .sort('timestamp')
    .drop(['is_organic', 'len', 'item_id'])
)

# 2. Оставляем последние события пользователя
train_events = (
    train_events
    .group_by('uid', maintain_order=True)
    .tail(MAX_EVENTS_PER_USER - 1)
)

# 3. Добавляем BOS=0 и собираем истории
train_histories = (
    pl.concat([
        (
            train_events.select('uid').unique()
            .with_columns(pl.lit(0, dtype=pl.UInt32).alias('token_id'))
        ),
        train_events.drop('timestamp')
    ])
    .group_by('uid', maintain_order=True)
    .agg('token_id')
)


In [9]:
tests.test_train_data(train_events, train_histories)

### 5. Датасет для обучения

Идея: мы превращаем таблицу с историями (`uid -> list[token_id]`) в поток **непрерывных** токенов и режем его на батчи длиной `batch_size * seq_len + 1`.

#### Что важно понять

* `__iter__` — это генератор: он **возвращает батчи через `yield`**.
* В начале каждой эпохи (каждого `__iter__`) можно перемешать пользователей:
  `df.sample(fraction=1.0, shuffle=True, seed=self.seed)`
* Паддинг **не делаем**: просто берём подряд идущие токены из потока.
* В один батч могут попасть куски разных пользователей — это ок, потому что **BOS=0** разделяет истории.
* На один батч нужно `batch_size * seq_len + 1` токен:

  * `inputs` — первые `batch_size * seq_len`
  * `targets` — те же токены, но сдвинутые на 1
  * `size = batch_size * seq_len`


In [10]:
from dataclasses import dataclass

import numpy as np
import polars as pl
import torch


@dataclass
class TrainingBatch:
    inputs: torch.Tensor
    targets: torch.Tensor
    size: int


class TrainingDataset:
    def __init__(
            self,
            df: pl.DataFrame,
            batch_size: int,
            seq_len: int,
            device: str = "cuda",
            chunk_rows: int = 64_000,
            shuffle: bool = True,
            seed: int | None = 42,
            pin_memory: bool = True
    ):
        self.df = df
        self.batch_size = batch_size
        self.seq_len = seq_len
        self.device = device
        self.chunk_rows = chunk_rows
        self.shuffle = shuffle
        self.seed = seed
        self.pin_memory = pin_memory

        self.batch_num_tokens = batch_size * seq_len + 1
        self.total_num_tokens = int(self.df.get_column("token_id").list.len().sum())

    def __len__(self):
        return self.total_num_tokens // self.batch_num_tokens

    def __iter__(self):
        df = self.df

        if self.shuffle:
            df = df.sample(fraction=1, shuffle=True, seed=self.seed)

        buf = np.ndarray((0,), dtype=np.int64)
        for i in range(0, df.height // self.chunk_rows + 1):
            chunk = df.slice(i * self.chunk_rows,
                             min(df.height - i * self.chunk_rows, self.chunk_rows))
            tokens = chunk.explode('token_id').get_column('token_id').cast(int).to_numpy()
            buf = np.concat((buf, tokens))
            while len(buf) >= self.batch_num_tokens + 1:
                t_cpu = torch.as_tensor(buf[:self.batch_num_tokens], device='cpu')
                if self.pin_memory:
                    t_cpu = t_cpu.pin_memory()
                t = t_cpu.to(self.device)
                inputs = t[:-1].view((self.batch_size, self.seq_len))
                targets = t[1:].view((self.batch_size, self.seq_len))
                yield TrainingBatch(inputs=inputs, targets=targets, size=inputs.numel())
                buf = buf[self.batch_num_tokens:]


In [11]:
dataloader = TrainingDataset(train_histories, batch_size=32, seq_len=100, shuffle=True, device='cuda')

assert len(dataloader) == 1128
batch = next(iter(dataloader))
assert batch.inputs.shape == batch.targets.shape == (32, 100)
assert batch.size == 100 * 32

### 6. Трансформер (энкодер GPT-style)

Здесь вы реализуете **минимальный GPT-энкодер** для последовательностей токенов.

#### Что должно получиться

* Вход: `token_ids` размера `(B, T)`
* Выход энкодера: `x` размера `(B, T, d_model)`
* (Дальше отдельно можно повесить `head` и получить logits `(B, T, vocab_size)`.)


#### 1) `CausalSelfAttention` (у вас уже готово)

Ключевое:

* `qkv = Linear(d_model → 3*d_model)`
* reshape в multi-head
* `F.scaled_dot_product_attention(..., attn_mask=None, is_causal=True)`
* `proj = Linear(d_model → d_model)`

#### 2) `MLP`

Должно быть:

* `Linear(d_model → 4*d_model)`
* `gelu`
* `dropout`
* `Linear(4*d_model → d_model)`


#### 3) `Block` (Pre-LN)

Формула:

* `x = x + dropout(attn(LN(x)))`
* `x = x + dropout(mlp(LN(x)))`

(Один `Dropout` можно переиспользовать.)


#### 4) `GPT` (эмбеддинги + блоки + финальный LN)

* `tok_emb = nn.Embedding(vocab_size, d_model)`
* `pos_emb = nn.Embedding(max_seq_len, d_model)`
* `x = tok_emb(token_ids) + pos_emb(positions)`
* dropout → `n_layers` блоков → `ln_f`

Важно: позиции делаем `0..T-1`, и **T не должен превышать max_seq_len**.


In [12]:
import torch
import torch.nn.functional as F
from torch import nn, Tensor


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.qkv = nn.Linear(d_model, 3 * d_model, bias=True)
        self.proj = nn.Linear(d_model, d_model, bias=True)
        self.dropout = dropout

    def forward(self, x: Tensor) -> Tensor:
        B, T, D = x.shape

        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=None,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,
        )

        y = y.transpose(1, 2).contiguous().view(B, T, D)
        y = self.proj(y)
        return y


class MLP(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.0):
        super().__init__()

        self.fc1 = torch.nn.Linear(d_model, 4 * d_model)

        self.fc2 = torch.nn.Linear(4 * d_model, d_model)

        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x: Tensor) -> Tensor:
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


class Block(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()

        self.ln1 = torch.nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.dropout = torch.nn.Dropout(dropout)

        self.ln2 = torch.nn.LayerNorm(d_model)
        self.mlp = MLP(d_model, dropout)

    def forward(self, x: Tensor) -> Tensor:
        x = x + self.dropout(self.attn(self.ln1(x)))
        x = x + self.dropout(self.mlp(self.ln2(x)))
        return x


class GPT(nn.Module):
    def __init__(
            self,
            vocab_size: int,
            max_seq_len: int,
            n_layers: int,
            d_model: int,
            n_heads: int,
            dropout: float = 0.0,
    ):
        super().__init__()

        self.max_seq_len = max_seq_len

        self.tok_emb = torch.nn.Embedding(vocab_size, d_model)

        self.pos_emb = torch.nn.Embedding(max_seq_len, d_model)

        self.drop = torch.nn.Dropout(dropout)

        self.blocks = torch.nn.ModuleList(
            Block(d_model, n_heads, dropout) for _ in range(n_layers)
        )

        self.ln_f = torch.nn.LayerNorm(d_model)

        self.head = torch.nn.Linear(d_model, vocab_size)

    def forward(self, token_ids: Tensor) -> Tensor:
        B, T = token_ids.shape
        assert T <= self.max_seq_len

        device = next(self.parameters()).device
        token_ids = token_ids.to(device)

        pos = torch.arange(T, device=device)

        x = self.tok_emb(token_ids) + self.pos_emb(pos)
        x = self.drop(x)

        for blk in self.blocks:
            x = blk(x)

        x = self.ln_f(x)
        return x

### 7. Граф вычислений (готовая реализация)
Оборачиваем трансформер, голову и подсчет лосса в единый граф вычислений:

In [13]:

class Graph(nn.Module):
    def __init__(
            self,
            vocab_size: int,
            max_seq_len: int,
            n_layers: int = 4,
            d_model: int = 256,
            n_heads: int = 4,
            dropout: float = 0.0
    ):
        super().__init__()
        self.gpt = GPT(
            vocab_size=vocab_size,
            max_seq_len=max_seq_len,
            n_layers=n_layers,
            d_model=d_model,
            n_heads=n_heads,
            dropout=dropout,
        )
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, batch: TrainingBatch):
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            x = self.gpt(batch.inputs)
            logits = self.head(x)

        return F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            batch.targets.view(-1)
        )

### 8. Цикл обучения (готовая реализация)

Это наш основной механизм обучения - он гоняет батчи, считает лосс, делает `backward()`, обновляет веса, иногда валидируется и всё логирует.

#### Главные сущности

* `graph` — модель (у вас она умеет: `loss = graph(batch)`)
* `train_dataloader` — источник батчей `TrainingBatch(inputs, targets, size)`
* `optimizer` — обновляет параметры модели
* `scheduler` — меняет learning rate по шагам (опционально)
* `validation_func()` — считает метрики на валидации (опционально)
* `SummaryWriter` — пишет логи для TensorBoard

#### Что происходит по шагам

1. **Эпохи и батчи**

* внешний цикл `for epoch in range(num_epochs)`
* внутри идём по батчам `range(len(train_dataloader))`

2. **Валидация раз в `eval_every` шагов**

* если `validation_func` задан и `eval_every != -1`
* запускается при `global_step % eval_every == 0` или на самом последнем шаге
* делаем `graph.eval()` + `torch.inference_mode()` → считаем метрики → обратно `graph.train()`
* пишем метрики в tensorboard: `valid/<name>`

3. **Один train-step**

* `optimizer.zero_grad(set_to_none=True)` — обнуляем градиенты
* `grad_accum_steps` раз:

  * берём батч `batch = next(train_it)`
  * считаем `loss = graph(batch)` под mixed precision (`autocast(bfloat16)`)
  * делим лосс на `grad_accum_steps` (чтобы суммарный градиент был как от большого батча)
  * `loss.backward()`

4. **Логируем train loss**

* `writer.add_scalar("train/loss", train_loss, tokens_passed)`

5. **Gradient clipping (опционально)**

* если `grad_clip > 0`: `clip_grad_norm_`
* логируем `optim/grad_norm`

6. **Шаг оптимизатора + scheduler**

* `optimizer.step()`
* если есть `scheduler`: `scheduler.step()` и логируем `optim/lr`

7. **Скорость**

* меряем время шага
* логируем:

  * `time/train_step_time(s)`
  * `time/tokens_per_sec(k)` (тысяч токенов/сек)


#### Важные детали, на которые стоит обратить внимание

* `tokens_passed` — общий счётчик токенов (ось X в TensorBoard); удобно сравнивать ран по “сколько данных увидели”, а не по “сколько шагов”.
* `curr_tokens` — сколько токенов обработали именно на этом шаге (с учётом grad accumulation).
* `torch.cuda.synchronize()` нужен, чтобы честно мерять время (GPU асинхронный).
* `autocast("cuda", torch.bfloat16)` — ускоряет и экономит память (если модель/железо поддерживает).


In [14]:
import time
import tqdm
import torch
from torch.utils.tensorboard import SummaryWriter


def train_loop(
        graph,
        train_dataloader,
        log_dir: str,
        optimizer,
        scheduler=None,
        num_epochs: int = 1,
        grad_accum_steps: int = 1,
        grad_clip: float = 0.0,
        eval_every: int = -1,
        validation_func=None,
):
    graph.train()
    writer = SummaryWriter(log_dir=log_dir)

    tokens_passed = 0
    global_step = 0

    for epoch in range(num_epochs):
        train_it = iter(train_dataloader)

        num_batches = len(train_dataloader)

        for batch_idx in tqdm.tqdm(range(num_batches), total=num_batches, desc=f"epoch {epoch}"):

            last_step = (epoch == num_epochs - 1) and (batch_idx == num_batches - 1)

            if validation_func is not None and eval_every != -1:
                if global_step % eval_every == 0 or last_step:
                    graph.eval()
                    with torch.inference_mode():
                        metrics = validation_func()
                    graph.train()

                    for name, value in metrics.items():
                        if torch.is_tensor(value):
                            value = float(value.detach().cpu())
                        writer.add_scalar(f"valid/{name}", value, tokens_passed)

            torch.cuda.synchronize()
            train_step_t0 = time.time()

            optimizer.zero_grad(set_to_none=True)

            train_loss = 0.0
            curr_tokens = 0

            for _ in range(grad_accum_steps):
                batch = next(train_it)
                curr_tokens += batch.size

                with torch.autocast("cuda", torch.bfloat16):
                    loss = graph(batch)

                loss = loss / grad_accum_steps
                train_loss += float(loss.detach())
                loss.backward()

            tokens_passed += curr_tokens

            writer.add_scalar("train/loss", train_loss, tokens_passed)

            if grad_clip > 0.0:
                grad_norm = torch.nn.utils.clip_grad_norm_(graph.parameters(), grad_clip)
                writer.add_scalar("optim/grad_norm", float(grad_norm.detach().cpu()), tokens_passed)

            optimizer.step()
            if scheduler is not None:
                scheduler.step()
                writer.add_scalar("optim/lr", optimizer.param_groups[0]["lr"], tokens_passed)

            torch.cuda.synchronize()
            train_step_t1 = time.time()

            dt = train_step_t1 - train_step_t0
            writer.add_scalar("time/train_step_time(s)", dt, tokens_passed)
            writer.add_scalar("time/tokens_per_sec(k)", (curr_tokens / dt) // 1000, tokens_passed)

            global_step += 1

    writer.close()

### 10. Запуск обучения

Здесь непосредственно запускаем обучение. На его прогресс можно (и нужно) посмотреть с помощью `tensorboard --logdir logs`

In [16]:
graph = Graph(
    vocab_size=30001,
    max_seq_len=100,
    n_layers=4,
    dropout=0.1
).cuda()
compiled_graph = torch.compile(graph, mode="default")

optimizer = torch.optim.AdamW(
    compiled_graph.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
    fused=True
)

train_loop(
    compiled_graph,
    num_epochs=5,
    train_dataloader=dataloader,
    log_dir='logs/test',
    optimizer=optimizer,
    grad_clip=1.0,
    grad_accum_steps=1
)

epoch 4: 100%|██████████| 1128/1128 [02:14<00:00,  8.41it/s]


In [17]:
%load_ext tensorboard
%tensorboard --logdir logs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


<IPython.core.display.Javascript object>

### 11. Подготовка данных для оценки качества

Цель: для каждого пользователя из `test` собрать **историю из трейна**, чтобы потом на ней делать предсказание следующего айтема.

#### Что делаем

1. Берём множество тестовых пользователей:

* `test_users = test.select('uid').unique()`

2. Для этих пользователей берём их события из `train_events` (уже токенизированные):

* фильтруем по `uid` через `semi join`
* сортируем по `timestamp`
* для каждого `uid` оставляем последние `MAX_LEN_PER_USER - 2` токенов
  (потому что 1 токен — BOS, и обычно ещё 1 “слот” оставляем под предсказание)

3. Добавляем `BOS=0` в начало каждой истории:

* конкатим `[uid, BOS]` + `[uid, token_id events]`
* группируем в список `token_id` с `maintain_order=True`

4. (опционально) считаем длину истории и сортируем по длине — удобно для дебага.


In [18]:
MAX_LEN_PER_USER = 100  # итоговая длина истории, включая BOS
BOS = 0

test_users = (
    test
    .select("uid")
    .unique()
)

test_user_events = (
    train_events
    .join(test_users, on="uid", how="semi")
    .sort("timestamp")
    .group_by("uid", maintain_order=True)
    .tail(n=MAX_LEN_PER_USER - 2)
    .select(["uid", "token_id"])
)

test_histories = (
    pl.concat([
        test_users.with_columns(pl.lit(0, dtype=pl.UInt32).alias('token_id')),
        test_user_events
    ])
    .group_by('uid', maintain_order=True)
    .agg('token_id')
)

test_histories = (
    test_histories
    .with_columns(length=pl.col("token_id").list.len())
    .sort("length", descending=True)
)

In [19]:
assert test_users.height == test_histories.height == 37446


### 12. Тестовый датасет

* в train мы выдавали непрерывный поток токенов
* в test мы работаем с **отдельными историями пользователей**
* поэтому здесь нужен **padding внутри батча**



In [20]:
from dataclasses import dataclass

import polars as pl
import torch
from torch.nn.utils.rnn import pad_sequence


@dataclass
class TestBatch:
    token_ids: torch.Tensor  # [B, T_max]
    lengths: torch.Tensor  # [B]


class TestDataset:
    def __init__(
            self,
            df: pl.DataFrame,
            batch_size: int,
            device: str = "cuda",
    ):
        self.df = df
        self.batch_size = batch_size
        self.device = device

        self._tensors = [torch.tensor(i) for i in df['token_id']]

    def __len__(self):
        return math.ceil(len(self._tensors) / self.batch_size)

    def __iter__(self):
        for i in range(len(self)):
            seqs = self._tensors[i * self.batch_size:
                        min((i + 1) * self.batch_size, len(self._tensors))]

            batch = pad_sequence(
                seqs,
                batch_first=True
            )
            lengths = torch.tensor([len(j) for j in seqs])
            yield TestBatch(batch, lengths)


## 13. Оценка качества

Идея:

1. прогоняем истории через GPT → получаем hidden states
2. берём hidden state **последнего реального токена** (по `lengths-1`)
3. через `head` получаем logits по словарю
4. берём topK токенов-кандидатов, маппим обратно в `item_id`
5. считаем метрики `evaluate(...)`

In [21]:
import torch
import polars as pl

TOPK = 100

ds = TestDataset(test_histories, batch_size=128)

with torch.inference_mode():
    all_candidates = []

    for batch in tqdm.tqdm(ds):
        with torch.autocast(device_type='cuda', dtype=torch.float):
            hidden_states = graph.gpt(batch.token_ids)
            last_hidden_state = hidden_states[
                torch.arange(len(batch.lengths)),
                batch.lengths - 1
            ]
            logits = graph.head(last_hidden_state)

        logits[:, 0] = -torch.inf

        _, indices = torch.topk(logits, k=TOPK)
        all_candidates.append(indices.cpu())

candidates = torch.cat(all_candidates, dim=0)

candidates_df = pl.DataFrame({"uid": test_histories['uid'], "token_id": candidates})

candidates_df = candidates_df.explode("token_id")

candidates_df = candidates_df.join(item_to_token_id, on="token_id", how="left")

candidates_df = candidates_df.group_by("uid", maintain_order=True).agg(pl.col("item_id"))

evaluate(
    targets=test_targets,
    candidates=dict(candidates_df.iter_rows()),
    catalog_size=VOCAB_ITEMS,
    topk=TOPK
)

100%|██████████| 293/293 [00:07<00:00, 41.76it/s] 


{'hitrate': 0.3318378464989585,
 'recall': 0.10997727333187453,
 'ndcg': 0.04393420372449459,
 'coverage': 0.6958666666666666}